# 01 · Data Quality & Cleaning

Lecture 1 — From Messy Medical Data to Statistical Insight

*Created by Master of AI.*

> **Synthetic educational data only.** This training is not intended for diagnosis, treatment, triage, medication decisions, or clinical decision-making.

## 1. Load the raw file *as text*
Reading every column as a string shows the data exactly as delivered.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
DATA = "../data/"
raw = pd.read_csv(DATA + "messy_patient_screening_data.csv", dtype=str, keep_default_na=False)
raw.shape

In [ ]:
raw.sample(8, random_state=1)

## 2. Completeness — missing values *including placeholder text*

In [ ]:
tokens = {"", "na", "n/a", "nan", "null", "none", "?", "-", "missing", "not recorded", "unknown"}
missing = raw.apply(lambda s: s.str.strip().str.lower().isin(tokens))
(missing.mean() * 100).round(2).sort_values(ascending=False)

## 3. Uniqueness — duplicates

In [ ]:
print("exact duplicates:", raw.duplicated().sum())
norm = raw.apply(lambda s: s.str.strip().str.lower())
print("patient_id + visit_date duplicates:", norm.duplicated(subset=["patient_id", "visit_date"]).sum())

## 4. Consistency — one concept, many spellings

In [ ]:
for col in ["sex", "smoking_status", "family_history_flag", "risk_group"]:
    print(col, sorted(raw[col].unique()))

## 5. Validity — numbers stored as text, impossible values

In [ ]:
numeric_like = raw["fasting_glucose"].str.fullmatch(r"-?\d+(\.\d+)?")
raw.loc[~numeric_like & (raw["fasting_glucose"] != ""), "fasting_glucose"].value_counts().head(10)

In [ ]:
age = pd.to_numeric(raw["age"].str.extract(r"(-?\d+)")[0], errors="coerce")
age[(age < 18) | (age > 100)].value_counts()

## 6. Cleaning rules (same logic as `python/medlib.py`)

In [ ]:
def extract_number(v):
    m = pd.Series([v]).astype(str).str.extract(r"(-?\d+(?:[.,]\d+)?)")[0].iloc[0]
    return np.nan if pd.isna(m) else float(m.replace(",", "."))

def parse_bp(v):
    if "/" in str(v):
        s, d = str(v).split("/", 1)
        return extract_number(s), extract_number(d)
    return extract_number(v), np.nan

print(parse_bp("138/88"), extract_number("110 mg/dL"), extract_number("6.2%"), extract_number("27,4"))

## 7. Compare with the clean file produced by `python/02_cleaning.py`

In [ ]:
clean = pd.read_csv(DATA + "clean_patient_screening_data.csv")
print("raw rows:", len(raw), " clean rows:", len(clean))
print(clean["sex"].value_counts(dropna=False))
print(clean["smoking_status"].value_counts(dropna=False))

**Key idea:** impossible values become *missing* — they are never silently “fixed”. Plausible extremes are kept and flagged in `outliers_patient_data.csv`.